In [ ]:
df_products_raw = (spark.read
                        .option("multiline", "true")
                        .json("Files/raw/products/products.json")
)

In [ ]:
display(df_products_raw)

In [ ]:
df_products_raw.printSchema()

In [ ]:
expected_top_level = {"products", "total", "skip", "limit"}

actual_top_level = set(df_products_raw.columns)

missing_columns = expected_top_level - actual_top_level

if missing_columns:
    raise ValueError(
        f"Bronze schema validation failed. Missing columns: {missing_columns}"
    )

print("Bronze schema validation passed.")

In [ ]:
from pyspark.sql import functions as F

df_products_exploded = (df_products_raw
.select(F.explode("products").alias("product"))
)

display(df_products_exploded)


In [ ]:
raw_product_count = df_products_exploded.count()

if raw_product_count == 0:
    raise ValueError("No product records found after exploding Bronze data.")

print(f"Product records found: {raw_product_count}")

In [ ]:
df_products_flat = (
    df_products_exploded
    .select("product.*")
)

display(df_products_flat)

In [ ]:
df_products_silver = (
    df_products_flat
    .select(
        "id",
        "title",
        "description",
        "category",
        "brand",
        "sku",
        "price",
        "discountPercentage",
        "rating",
        "stock",
        "availabilityStatus",
        "minimumOrderQuantity",
        "returnPolicy",
        "warrantyInformation",
        "shippingInformation",
        F.col("dimensions.width").alias("width"),
        F.col("dimensions.height").alias("height"),
        F.col("dimensions.depth").alias("depth"),
        F.col("meta.barcode").alias("barcode"),
        F.col("meta.createdAt").alias("created_at"),
        F.col("meta.updatedAt").alias("updated_at")
    )
)

display(df_products_silver)

In [ ]:
#data quality checks

# # Null product IDs
# df_products_silver.filter(
#     F.col("id").isNull()
# ).count()

# Null product brand
df_products_silver.filter(
    F.col("brand").isNull()
).count()


In [10]:
from pyspark.sql import functions as F

df_products_silver = (df_products_silver
                      .withColumn("brand", 
                     F.coalesce(F.col("brand"), F.lit("unknown"))
                )
        )

StatementMeta(, c7dbfec6-0244-4916-97ce-4ebe234ba77f, 12, Finished, Available, Finished, False)

In [ ]:
# Duplicate product IDs
df_products_silver.groupBy("id") \
    .count() \
    .filter(F.col("count") > 1) \
    .show()

In [ ]:
# Invalid numeric/business values
df_products_silver.filter(
    (F.col("price") < 0) |
    (F.col("stock") < 0) |
    (F.col("rating") < 0) |
    (F.col("rating") > 5)
).show()

In [ ]:
df_products_silver.select(
    *[
        F.sum(F.col(c).isNull().cast("int")).alias(c)
        for c in [
            "id",
            "title",
            "category",
            "brand",
            "price",
            "stock",
            "rating"
        ]
    ]
).show()

In [ ]:
null_id_count = (
    df_products_silver
    .filter(F.col("id").isNull())
    .count()
)

duplicate_id_count = (
    df_products_silver
    .groupBy("id")
    .count()
    .filter(F.col("count") > 1)
    .count()
)

invalid_business_values = (
    df_products_silver
    .filter(
        (F.col("price") < 0) |
        (F.col("stock") < 0) |
        (F.col("rating") < 0) |
        (F.col("rating") > 5)
    )
    .count()
)

if null_id_count > 0:
    raise ValueError(f"Data quality failed: {null_id_count} null product IDs.")

if duplicate_id_count > 0:
    raise ValueError(
        f"Data quality failed: {duplicate_id_count} duplicate product IDs."
    )

if invalid_business_values > 0:
    raise ValueError(
        f"Data quality failed: {invalid_business_values} invalid business values."
    )

print("Critical product data-quality checks passed.")

In [ ]:
(
    df_products_silver
    .write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("lh_retail_silver.dbo.products")
)

In [ ]:
from pyspark.sql import functions as F

df_reviews_exploded = (
    df_products_flat
    .select(
        F.col("id").alias("product_id"),
        F.explode("reviews").alias("customer_review")
    )
)

display(df_reviews_exploded)

In [ ]:
df_reviews_flat = (
    df_reviews_exploded
    .select(
        "product_id",
        "customer_review.*"
    )
)

display(df_reviews_flat)

In [ ]:
df_reviews_silver = (
    df_reviews_flat
    .select(
        "product_id",
        F.col("rating").alias("review_rating"),
        "comment",
        F.col("date").alias("review_date"),
        F.col("reviewerName").alias("reviewer_name"),
        F.col("reviewerEmail").alias("reviewer_email")
    )
)

In [ ]:
# Null product IDs
df_reviews_silver.filter(
    F.col("product_id").isNull()
).count()

In [ ]:
# Invalid review ratings
df_reviews_silver.filter(
    (F.col("review_rating") < 1) |
    (F.col("review_rating") > 5)
).show()

In [ ]:
# Null summary
df_reviews_silver.select(
    *[
        F.sum(F.col(c).isNull().cast("int")).alias(c)
        for c in [
            "product_id",
            "review_rating",
            "comment",
            "review_date",
            "reviewer_name",
            "reviewer_email"
        ]
    ]
).show()

In [ ]:
(
    df_reviews_silver
    .write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("lh_retail_silver.dbo.product_reviews")
)